# Analyse des données de MovieLens

In [7]:
# 📚 Chargement des librairies nécessaires
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import requests
import pickle
import time
from pathlib import Path
from typing import Dict, Optional
from tqdm import tqdm
from dotenv import load_dotenv

print("✅ Imports réussis!")

✅ Imports réussis!


In [ ]:
# Chargement des données
movies = pd.read_csv('../data/movies.csv')
ratings = pd.read_csv('../data/ratings.csv')
tags = pd.read_csv('../data/tags.csv')


In [ ]:
import pandas as pd

links = pd.read_csv('../data/links.csv')

# Extraire l'ID du film
toy_story = links[links.movieId == 1]
imdb_id = toy_story.imdbId.values[0]  # Extraire AVANT la f-string
tmdb_id = toy_story.tmdbId.values[0]

# Construire les URLs correctement
imdb_url = f"https://imdb.com/title/tt{str(int(imdb_id)).zfill(7)}/"
tmdb_url = f"https://www.themoviedb.org//movie/{int(tmdb_id)}"

print(f"URL IMDb: {imdb_url}")
print(f"URL TMDb: {tmdb_url}")

# Récupérer les données de la page IMDb
import requests
from bs4 import BeautifulSoup
response = requests.get(imdb_url)


URL IMDb: https://imdb.com/title/tt0114709/
URL TMDb: https://www.themoviedb.org//movie/862


In [8]:
# Charger la clé API depuis .env
load_dotenv()

# Configuration
TMDB_API_KEY = os.getenv('TMDB_API_KEY')
TMDB_BASE_URL = "https://www.themoviedb.org//movie/"

# Paramètres
API_LANGUAGE = 'fr-FR'          # 'fr-FR' ou 'en-US'
API_DELAY = 0.25                # Délai entre requêtes (4 req/s)
MAX_ACTORS = 5                  # Nombre d'acteurs à récupérer
MAX_KEYWORDS = 10               # Nombre de mots-clés

# Chemins (adapter selon ton organisation)
DATA_DIR = Path('../data')      # ou Path('./data') si même dossier
CACHE_FILE = DATA_DIR / 'tmdb_cache.pkl'
OUTPUT_FILE = DATA_DIR / 'movies_enriched.csv'

# Vérifier la clé API
if not TMDB_API_KEY:
    print("❌ TMDB_API_KEY non trouvée!")
    print("Crée un fichier .env avec: TMDB_API_KEY=ta_cle_api")
else:
    print(f"✅ Configuration OK")
    print(f"🔑 API Key: {'*' * (len(TMDB_API_KEY) - 4) + TMDB_API_KEY[-4:]}")


✅ Configuration OK
🔑 API Key: ****************************1741


In [ ]:
import requests
import json

# Ta clé API TMDb
TMDB_API_KEY = os.getenv('TMDB_API_KEY')

# ID TMDb d'un film (exemple: Toy Story = 862)
tmdb_id = 862

# Faire l'appel API
url = f"https://api.themoviedb.org/3/movie/{tmdb_id}"
params = {
    'api_key': TMDB_API_KEY,
    'language': 'fr-FR',
    'append_to_response': 'credits,keywords,external_ids,videos'
}

response = requests.get(url, params=params)
data = response.json()

# Afficher tout ce qui est disponible
print("="*70)
print(f"FILM : {data.get('title')}")
print("="*70)

print("\n📋 INFOS DE BASE:")
print(f"  Titre original: {data.get('original_title')}")
print(f"  Synopsis: {data.get('overview')}")
print(f"  Date sortie: {data.get('release_date')}")
print(f"  Durée: {data.get('runtime')} min")
print(f"  Statut: {data.get('status')}")

print("\n💰 FINANCIER:")
print(f"  Budget: ${data.get('budget'):,}")
print(f"  Revenus: ${data.get('revenue'):,}")

print("\n⭐ POPULARITÉ:")
print(f"  Note: {data.get('vote_average')}/10")
print(f"  Votes: {data.get('vote_count')}")
print(f"  Popularité: {data.get('popularity')}")

print("\n🎭 GENRES:")
for genre in data.get('genres', []):
    print(f"  • {genre['name']}")

print("\n🎬 ÉQUIPE:")
crew = data.get('credits', {}).get('crew', [])
directors = [c['name'] for c in crew if c['job'] == 'Director']
print(f"  Réalisateur(s): {', '.join(directors)}")

print("\n👥 ACTEURS (top 5):")
for actor in data.get('credits', {}).get('cast', [])[:5]:
    print(f"  • {actor['name']} ({actor['character']})")

print("\n🏷️ MOTS-CLÉS:")
for keyword in data.get('keywords', {}).get('keywords', [])[:10]:
    print(f"  • {keyword['name']}")

print("\n🏢 PRODUCTION:")
for company in data.get('production_companies', []):
    print(f"  • {company['name']}")

print("\n🌍 PAYS:")
for country in data.get('production_countries', []):
    print(f"  • {country['name']}")

print("\n🔗 IDs EXTERNES:")
external = data.get('external_ids', {})
print(f"  IMDb: {external.get('imdb_id')}")
print(f"  Facebook: {external.get('facebook_id')}")

print("\n🖼️ IMAGES:")
print(f"  Poster: https://image.tmdb.org/t/p/w500{data.get('poster_path')}")
print(f"  Backdrop: https://image.tmdb.org/t/p/original{data.get('backdrop_path')}")

print("\n📦 TOUTES LES CLÉS DISPONIBLES:")
print(list(data.keys()))


FILM : Toy Story

📋 INFOS DE BASE:
  Titre original: Toy Story
  Synopsis: Dans un monde où les jouets vivent leur vie quand les humains ne sont pas présents, Toy Story emmène...
  Date sortie: 1995-11-22
  Durée: 81 min
  Statut: Released

💰 FINANCIER:
  Budget: $30,000,000
  Revenus: $401,157,969

⭐ POPULARITÉ:
  Note: 8.0/10
  Votes: 19519
  Popularité: 19.9716

🎭 GENRES:
  • Familial
  • Comédie
  • Animation
  • Aventure

🎬 ÉQUIPE:
  Réalisateur(s): John Lasseter

👥 ACTEURS (top 5):
  • Tom Hanks (Woody (voice))
  • Tim Allen (Buzz Lightyear (voice))
  • Don Rickles (Mr. Potato Head (voice))
  • Jim Varney (Slinky Dog (voice))
  • Wallace Shawn (Rex (voice))

🏷️ MOTS-CLÉS:
  • rescue
  • friendship
  • mission
  • jealousy
  • villain
  • bullying
  • elementary school
  • rivalry
  • anthropomorphism
  • friends

🏢 PRODUCTION:
  • Pixar

🌍 PAYS:
  • United States of America

🔗 IDs EXTERNES:
  IMDb: tt0114709
  Facebook: pixartoystory

🖼️ IMAGES:
  Poster: https://image.tmdb.org/t